# RSNA Knee — ensemble submission (probe #3)

**3-member ensemble.** Members, per-label blend method, and the TTA decision are all loaded from a **frozen, versioned recipe file** (`src/rsna_knee/ensemble_recipe_v1.yaml`, bundled in the `rsna-knee-code` dataset) — nothing is hardcoded in this notebook. A future recipe change is a new version of that file plus one path update, not a notebook edit.

Recipe v1 (2026-09-09/10, `scripts/build_ensemble.py`): greedy forward selection on OOF-vs-`labels_v2_final` macro-AUC (gold-58 used only as a tripwire, never for selection) chose:
- `exp006` — DINOv2 ViT-B/14 on `labels_v3_candidate`
- `exp003` — DINOv2 ViT-B/14 on `labels_v2_final`
- `exp005` — DINOv2 ViT-S/14 on `labels_v3_candidate`

"rank-mean" at single-study inference time has no population to rank against, so it's computed as an empirical-CDF lookup against each member's own stored OOF score distribution (`rank_reference.npz`, `infer_core.ensemble_blend`) rather than silently degraded to prob-mean.

**TTA (laterality-swap dual-pass): tested and REJECTED**, recorded in the recipe file itself (`tta.enabled: false`) — real inference passes on all 3 members x 5 folds measured OOF macro -0.0014 with TTA vs without (8/12 labels regressed).

**Degradation ladder** (CLAUDE.md §9 — a time-budget calibration picks ONE rung for the whole run, not a per-study drop, so scoring stays consistent across the submission):
1. All 3 members (the top rung, TTA never included since it's rejected)
2. {exp006, exp005} (drop exp003 — the older-labels ViT-B, most redundant given exp006 is its v3-label successor)
3. Smallest model only: exp005 (ViT-S, fastest)

A short calibration pass on the first studies measures actual per-study cost, projects total runtime per rung, and picks the highest rung that fits the budget with margin — then the wall-clock guard (per-study prior fallback) stays active underneath as defense-in-depth in case the projection was wrong.

**Runtime is a PROJECTION** until run on the actual pinned accelerator — **holding this push until the accelerator is pinned and probe #1/#2 are through**, per instruction.

Expects attached Kaggle Datasets: `rsna-knee-code`, `rsna-knee-exp006-weights`, `rsna-knee-exp003-weights`, `rsna-knee-exp005-weights`, plus the competition data. Internet OFF.

In [ ]:
import importlib.util
import os, sys, time, traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch

def _find_dir(root: str, name_contains: str) -> Path | None:
    for p in Path(root).rglob("*"):
        if p.is_dir() and name_contains in p.name:
            return p
    return None

CODE_DIR = _find_dir("/kaggle/input", "rsna-knee-code")
TEST_DIR = _find_dir("/kaggle/input", "rsna-knee-abnormality-detection")
OUT_PATH = Path("/kaggle/working/submission.csv")

print("CODE_DIR:", CODE_DIR)
print("TEST_DIR:", TEST_DIR)
assert CODE_DIR is not None, "Attach rsna-knee-code"
assert TEST_DIR is not None, "Attach the competition data"

init_files = list(CODE_DIR.rglob("__init__.py"))
assert init_files, f"No __init__.py found under {CODE_DIR}"
pkg_dir = init_files[0].parent
spec = importlib.util.spec_from_file_location("rsna_knee", pkg_dir / "__init__.py", submodule_search_locations=[str(pkg_dir)])
rsna_knee = importlib.util.module_from_spec(spec)
sys.modules["rsna_knee"] = rsna_knee
spec.loader.exec_module(rsna_knee)

from rsna_knee.dataset import TARGET_COLUMNS, _patch_size_for_backbone
from rsna_knee.infer_core import (
    WallClockGuard, decode_study_slots, ensemble_blend, load_ensemble_recipe, nearest_patch_multiple,
    prior_row, run_study_inference, run_study_inference_tta,
)
from rsna_knee.model import SlotModel

# Frozen recipe (src/rsna_knee/ensemble_recipe_v1.yaml, bundled in the rsna-knee-code dataset) —
# members, per-label blend method, and the TTA decision all come from THIS file, not a hardcoded
# list here, so a future recipe version is a new file + one path change, not a notebook edit.
RECIPE_PATH = pkg_dir / "ensemble_recipe_v1.yaml"
assert RECIPE_PATH.exists(), f"Recipe file not found at {RECIPE_PATH} — check the rsna-knee-code dataset version"
loaded = load_ensemble_recipe(RECIPE_PATH)
MEMBERS = {name: {**spec} for name, spec in loaded["members"].items()}
RECIPE = loaded["recipe"]  # {label: (members_list, method)}
TTA_ACCEPTED = loaded["tta_enabled"]
for name, spec in MEMBERS.items():
    spec["dir"] = _find_dir("/kaggle/input", spec["kaggle_dataset"])
    print(name, "->", spec["dir"])
    assert spec["dir"] is not None and spec["dir"].exists(), f"Attach {spec['kaggle_dataset']}"

print(f"Recipe version {loaded['raw']['version']} ({loaded['raw']['created']}), "
      f"TTA_ACCEPTED={TTA_ACCEPTED}")
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())

In [ ]:
SLICES_PER_SERIES = 24
IMAGE_SIZE = 384
OUT_SIZE = nearest_patch_multiple(IMAGE_SIZE, _patch_size_for_backbone(next(iter(MEMBERS.values()))["backbone"]))  # 378 — same for all 3 members (all patch14)

GLOBAL_MEMBERS = list(MEMBERS.keys())

BUDGET_SECONDS = 8 * 3600  # 9h hard cap (CLAUDE.md §1) minus margin for scheduling/write
CALIBRATION_STUDIES = 10   # measure actual per-study cost on this many before picking a ladder rung

DEVICE = "cpu"
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    cap = major * 10 + minor
    supported_caps = [int(a[3:]) for a in torch.cuda.get_arch_list() if a.startswith("sm_")]
    if supported_caps and cap >= min(supported_caps):
        DEVICE = "cuda"
    else:
        print(f"  [WARN] GPU {torch.cuda.get_device_name(0)} (sm_{cap}) unsupported by this torch "
              f"build (min sm_{min(supported_caps) if supported_caps else '?'}) — CPU fallback.")
print("device:", DEVICE, "out_size:", OUT_SIZE, "TTA_ACCEPTED:", TTA_ACCEPTED)

In [ ]:
def load_member_models(spec: dict, device: str) -> list:
    models = []
    for fold_dir in sorted(spec["dir"].glob("fold*")):
        ckpt_path = fold_dir / "best.pt"
        if not ckpt_path.exists():
            print(f"  [WARN] no best.pt in {fold_dir}")
            continue
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model = SlotModel(spec["backbone"], num_targets=len(TARGET_COLUMNS), pretrained=False).to(device)
        model.load_state_dict(ckpt["ema"])
        model.eval()
        models.append(model)
    return models

member_models, member_rank_refs = {}, {}
for name, spec in MEMBERS.items():
    member_models[name] = load_member_models(spec, DEVICE)
    npz_path = spec["dir"] / "rank_reference.npz"
    member_rank_refs[name] = dict(np.load(npz_path)) if npz_path.exists() else {}
    print(f"{name}: {len(member_models[name])}/5 folds loaded, rank_reference "
          f"{'found' if npz_path.exists() else 'MISSING (rank-mean labels fall back to prob-mean)'}")

if all(len(m) == 0 for m in member_models.values()):
    raise RuntimeError("No member models loaded at all — cannot run inference.")

In [ ]:
test_df = pd.read_csv(TEST_DIR / "test.csv")
series_df = pd.read_csv(TEST_DIR / "test_series.csv").rename(
    columns={"StudyInstanceUID": "study_uid", "SeriesInstanceUID": "series_uid"}
)
series_root = TEST_DIR / "test_series"
print(f"{len(test_df)} test studies, {len(series_df)} series.")

In [ ]:
# Ladder rungs, cheapest-fallback last. Each rung = (use_tta, members_to_use). TTA is measured
# OOF-negative (frozen recipe file) so it never appears as an active rung, only smaller member sets.
LADDER = [
    ("no-TTA + 3 members", False, GLOBAL_MEMBERS),
    ("no-TTA + 2 members", False, ["exp006_dinov2b_slot_v3labels", "exp005_dinov2s_slot_v3labels"]),
    ("smallest model only", False, ["exp005_dinov2s_slot_v3labels"]),
]


def relative_cost(use_tta: bool, members: list[str]) -> float:
    """Rough relative inference cost vs. one ViT-S forward pass — ViT-B counted as 3x ViT-S
    (roughly matches parameter-count scaling), TTA doubles whichever members run."""
    unit = sum(3.0 if MEMBERS[m]["backbone"].startswith("vit_base") else 1.0 for m in members)
    return unit * (2.0 if use_tta else 1.0)


def run_member(name: str, slots, tag_sides, use_tta: bool) -> np.ndarray:
    models = member_models[name]
    if not models:
        return prior_row()
    if use_tta:
        return run_study_inference_tta(slots, tag_sides, models, DEVICE, SLICES_PER_SERIES, OUT_SIZE)
    return run_study_inference(slots, tag_sides, models, DEVICE, SLICES_PER_SERIES, OUT_SIZE)


def run_ensemble(slots, tag_sides, use_tta: bool, members: list[str]) -> np.ndarray:
    member_probs = {m: run_member(m, slots, tag_sides, use_tta) for m in members}
    # RECIPE (from the frozen recipe file) already gives {label: (members, method)} for the full
    # 3-member set; when the ladder has dropped to fewer members, use that label's method but
    # restrict to whichever of those members are actually active at this rung.
    active_recipe = {label: ([m for m in orig_members if m in members], method)
                      for label, (orig_members, method) in RECIPE.items()}
    return ensemble_blend(member_probs, active_recipe, member_rank_refs)

In [ ]:
guard = WallClockGuard(BUDGET_SECONDS)

# --- Calibration: run the full (top) rung on the first few studies, measure real per-study cost,
# project total time for every rung, pick the highest one that fits with margin. One rung is used
# for the WHOLE run (not switched mid-way) so every study is scored consistently.
n_calib = min(CALIBRATION_STUDIES, len(test_df))
calib_t0 = time.time()
calib_rows = []
for i, study_uid in enumerate(test_df["StudyInstanceUID"].iloc[:n_calib]):
    study_series = series_df[series_df["study_uid"] == study_uid].to_dict("records")
    slots, tag_sides = decode_study_slots(study_uid, study_series, series_root / study_uid, SLICES_PER_SERIES, OUT_SIZE)
    if slots:
        probs = run_ensemble(slots, tag_sides, *LADDER[0][1:])
        calib_rows.append({"StudyInstanceUID": study_uid, **{c: float(p) for c, p in zip(TARGET_COLUMNS, probs)}})
calib_elapsed = time.time() - calib_t0
per_study_at_rung0 = calib_elapsed / max(n_calib, 1)
rung0_cost = relative_cost(*LADDER[0][1:])
print(f"Calibration: {n_calib} studies, {calib_elapsed:.1f}s ({per_study_at_rung0:.2f}s/study at rung 0)")

chosen_rung_idx = 0
for idx, (label, use_tta, members) in enumerate(LADDER):
    projected_per_study = per_study_at_rung0 * (relative_cost(use_tta, members) / rung0_cost)
    projected_total = projected_per_study * len(test_df)
    fits = projected_total <= guard.budget * 0.85  # 15% margin under budget
    print(f"  rung {idx} ({label}): projected {projected_total / 60:.1f} min for {len(test_df)} studies "
          f"-> {'FITS' if fits else 'too slow'}")
    if fits:
        chosen_rung_idx = idx
        break
else:
    chosen_rung_idx = len(LADDER) - 1  # nothing fit comfortably — use the cheapest rung anyway

rung_label, USE_TTA, ACTIVE_MEMBERS = LADDER[chosen_rung_idx]
print(f"Chosen rung: {chosen_rung_idx} ({rung_label})")

rows = list(calib_rows) if chosen_rung_idx == 0 else []  # reuse calibration results only if they match the chosen rung
start_idx = len(rows)
n_prior_fallback = n_dual_flip = 0
for i in range(start_idx, len(test_df)):
    study_uid = test_df["StudyInstanceUID"].iloc[i]
    row = {"StudyInstanceUID": study_uid}
    try:
        if guard.over(margin=0.05):
            print(f"  [BUDGET] {guard.elapsed():.0f}s used at study {i}/{len(test_df)} — priors for the rest.")
            raise TimeoutError("wall-clock budget exhausted")
        study_series = series_df[series_df["study_uid"] == study_uid].to_dict("records")
        slots, tag_sides = decode_study_slots(study_uid, study_series, series_root / study_uid, SLICES_PER_SERIES, OUT_SIZE)
        if not slots:
            raise RuntimeError("no series decoded successfully for this study")
        if all(s not in ("R", "L") for s in tag_sides):
            n_dual_flip += 1
        probs = run_ensemble(slots, tag_sides, USE_TTA, ACTIVE_MEMBERS)
        row.update({c: float(p) for c, p in zip(TARGET_COLUMNS, probs)})
    except Exception as e:
        print(f"  [WARN] {study_uid}: {type(e).__name__}: {e} — using label priors")
        n_prior_fallback += 1
        row.update({c: float(p) for c, p in zip(TARGET_COLUMNS, prior_row())})
    rows.append(row)
    if (i + 1) % 100 == 0 or (i + 1) == len(test_df):
        print(f"  {i + 1}/{len(test_df)} studies, {guard.elapsed():.0f}s elapsed ({guard.fraction_used():.1%} of budget)")

print(f"Done: {len(rows)} studies, rung={chosen_rung_idx} ({rung_label}), {n_prior_fallback} full-prior fallbacks, "
      f"{n_dual_flip} dual-flip, {guard.elapsed():.0f}s total.")

In [ ]:
sub_df = pd.DataFrame(rows)[["StudyInstanceUID"] + TARGET_COLUMNS]

assert len(sub_df) == len(test_df), f"row count mismatch: {len(sub_df)} vs {len(test_df)}"
assert set(sub_df["StudyInstanceUID"]) == set(test_df["StudyInstanceUID"]), "study UID mismatch"
assert not sub_df[TARGET_COLUMNS].isna().any().any(), "NaNs in submission"
assert (sub_df[TARGET_COLUMNS] >= 0).all().all() and (sub_df[TARGET_COLUMNS] <= 1).all().all(), "probabilities out of [0,1]"
assert list(sub_df.columns) == ["StudyInstanceUID"] + TARGET_COLUMNS, f"header mismatch: {list(sub_df.columns)}"

sub_df.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH}: {len(sub_df)} rows, all checks passed. Rung used: {chosen_rung_idx} ({rung_label}).")
sub_df.head()